In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [2]:
from pathlib import Path

root = Path("/kaggle/input")

for p in root.rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/sithumbuddika/j26-se-361-qwen-server/kaggle_inference_server.py
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/train_pilot.jsonl
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/training_history.json
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/holdout_pilot.jsonl
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter/adapter_model.safetensors
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter/adapter_config.json
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter/README.md
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter/tokenizer.json
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter/tokenizer_config.json
/kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilo

In [3]:
from pathlib import Path

adapters = list(Path("/kaggle/input").rglob("adapter_config.json"))

print("Adapter configurations found:", len(adapters))

for p in adapters:
    print("Adapter location:", p.parent)

Adapter configurations found: 1
Adapter location: /kaggle/input/datasets/sithumbuddika/j26-se-361-qlora-adapter/qlora_pilot_backup/adapter


In [4]:
%pip install -q transformers peft accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -q --upgrade "bitsandbytes>=0.46.1" accelerate

Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import bitsandbytes as bnb
import importlib.metadata as metadata

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("bitsandbytes:", metadata.version("bitsandbytes"))
print("accelerate:", metadata.version("accelerate"))

GPU available: True
GPU: Tesla T4
bitsandbytes: 0.50.2
accelerate: 1.15.0


In [7]:
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

adapter_path = Path(
    "/kaggle/input/datasets/sithumbuddika/"
    "j26-se-361-qlora-adapter/"
    "qlora_pilot_backup/adapter"
)

assert (adapter_path / "adapter_config.json").exists()
assert (adapter_path / "adapter_model.safetensors").exists()
assert torch.cuda.is_available(), "GPU not available"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    base_model,
    str(adapter_path),
    is_trainable=False
)

model.eval()

print("Qwen2.5-Coder-7B-Instruct loaded successfully")
print("Existing QLoRA adapter loaded successfully")
print("GPU:", torch.cuda.get_device_name(0))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Qwen2.5-Coder-7B-Instruct loaded successfully
Existing QLoRA adapter loaded successfully
GPU: Tesla T4


In [8]:

import torch

messages = [
    {
        "role": "system",
        "content": (
            "Explain Java maintainability findings using only "
            "the supplied analyzer evidence. Do not invent metrics "
            "or claim behavioral equivalence."
        )
    },
    {
        "role": "user",
        "content": (
            "Explain this Java maintainability finding:\n"
            "Concern: HIGH_COMPLEXITY\n"
            "Method: Commander.updateQueue(QueueTask)\n"
            "Cyclomatic complexity: 14\n"
            "Preliminary configured threshold: 10\n"
            "Suggest a cautious refactoring direction."
        )
    }
]

encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

encoded = {
    key: value.to("cuda")
    for key, value in encoded.items()
}

model.eval()

with torch.inference_mode():
    outputs = model.generate(
        **encoded,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][encoded["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("=== REAL QWEN EXPLANATION ===")
print(response)


=== REAL QWEN EXPLANATION ===
The method `Commander.updateQueue(QueueTask)` has a cyclomatic complexity of 14, which is higher than the preliminary configured threshold of 10. This indicates that the method contains multiple paths through its code, making it more complex and harder to understand and maintain.

To address this concern, consider breaking down the method into smaller, more manageable methods. Each new method should have a lower cyclomatic complexity, ideally below the threshold. This will improve the readability and testability of the code, as well as make it easier for other developers to work with. Additionally, you could refactor the method by extracting repetitive code into separate methods or by simplifying conditional logic where possible.


In [9]:

%pip install -q fastapi uvicorn


Note: you may need to restart the kernel to use updated packages.


In [10]:

from pathlib import Path

scripts = list(
    Path("/kaggle/input").rglob("kaggle_inference_server.py")
)

assert len(scripts) == 1, f"Expected one server script, found {len(scripts)}"

print("Starting real Qwen inference server...")

exec(
    compile(
        scripts[0].read_text(encoding="utf-8"),
        str(scripts[0]),
        "exec"
    ),
    globals()
)


Starting real Qwen inference server...
Kaggle Qwen local API initialized at 127.0.0.1:8766

=== PRIVATE CONNECTION DETAILS: DO NOT SHARE TOKEN ===
TUNNEL URL: https://holds-settings-lexington-cancer.trycloudflare.com
ACCESS TOKEN: PjKlHI0I2eV9ZC6E6DGtic60kbvLpOQBwdgxtZmIMdg
Start windows_bridge.py on Windows, then click Generate explanation in VS Code.
Keep Kaggle notebook session active; endpoints are temporary.


In [20]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json

test_payload = {
    "task": "explain_maintainability_evidence",
    "schema_version": "1.1",
    "group": {
        "rank": 1,
        "ranking_explanation": "Preliminary ranking.",
        "findings": [
            {
                "concern": "HIGH_COMPLEXITY",
                "symbol": "Commander.updateQueue(QueueTask)",
                "rule_id": "METHOD_CYCLOMATIC_BOUNDARY",
                "evidence": {
                    "cyclomaticComplexity": "14",
                    "configuredMaxComplexity": "10"
                }
            }
        ]
    },
    "constraints": []
}

request = Request(
    "http://127.0.0.1:8766/explain",
    data=json.dumps(test_payload).encode(),
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer " + access_token
    },
    method="POST"
)

try:
    with urlopen(request, timeout=120) as response:
        print("HTTP status:", response.status)
        print(response.read().decode()[:3000])
except HTTPError as error:
    print("HTTP status:", error.code)
    print(error.read().decode()[:3000])
except Exception as error:
    print(type(error).__name__, str(error))

HTTP status: 200
{"explanation":"**Concern:** The method `Commander.updateQueue(QueueTask)` has a high cyclomatic complexity of 14, which exceeds the configured maximum of 10.\n\n**Evidence:**\n- Cyclomatic Complexity: 14\n- Configured Max Complexity: 10\n\n**Explanation:**\nHigh cyclomatic complexity indicates that the method is complex and difficult to understand, test, and maintain. It suggests that the method contains multiple paths through its logic, making it harder to reason about its behavior.\n\n**Refactoring Direction:**\nConsider breaking down the `updateQueue` method into smaller, more focused methods. This can reduce the complexity by isolating specific functionalities within each new method. Each smaller method should ideally have a lower cyclomatic complexity.\n\n**Independent Verification:**\nAfter refactoring, ensure that all unit tests for the original `updateQueue` method still pass. Additionally, consider writing new tests for the newly created smaller methods to co

In [21]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json

tunnel_url = input("Paste your Cloudflare tunnel URL: ").strip().rstrip("/")

request = Request(
    tunnel_url + "/explain",
    data=json.dumps(test_payload).encode("utf-8"),
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer " + access_token
    },
    method="POST"
)

try:
    with urlopen(request, timeout=120) as response:
        print("Tunnel HTTP status:", response.status)
        print(response.read().decode("utf-8")[:1500])
except HTTPError as error:
    print("Tunnel HTTP status:", error.code)
    print(error.read().decode("utf-8")[:1500])
except Exception as error:
    print(type(error).__name__, str(error))

Paste your Cloudflare tunnel URL:  https://holds-settings-lexington-cancer.trycloudflare.com


Tunnel HTTP status: 200
{"explanation":"**Concern:** The method `Commander.updateQueue(QueueTask)` has a high cyclomatic complexity of 14, which exceeds the configured maximum of 10.\n\n**Evidence:**\n- Cyclomatic Complexity: 14\n- Configured Max Complexity: 10\n\n**Explanation:**\nHigh cyclomatic complexity indicates that the method is complex and difficult to understand, test, and maintain. It suggests that the method contains multiple paths through its logic, making it harder to reason about its behavior.\n\n**Refactoring Direction:**\nConsider breaking down the `updateQueue` method into smaller, more focused methods. This can reduce the complexity by isolating specific functionalities within each new method. Each smaller method should ideally have a lower cyclomatic complexity.\n\n**Independent Verification:**\nAfter refactoring, ensure that all unit tests for the original `updateQueue` method still pass. Additionally, consider writing new tests for the newly created smaller method

In [23]:
import requests

try:
    r = requests.get(
        "http://127.0.0.1:8766/openapi.json",
        timeout=10
    )
    print("KAGGLE LOCAL API:", r.status_code)
except Exception as e:
    print("KAGGLE LOCAL API ERROR:", str(e))

KAGGLE LOCAL API: 404


In [24]:
import subprocess

result = subprocess.run(
    ["pgrep", "-af", "cloudflared"],
    capture_output=True,
    text=True
)

print(result.stdout or "No cloudflared process found")

390 /kaggle/working/cloudflared tunnel --url http://127.0.0.1:8766 --no-autoupdate



In [25]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Qwen model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Access token exists:", "access_token" in globals())

GPU available: True
Qwen model loaded: True
Tokenizer loaded: True
Access token exists: True
